# Ch.4 座学：機械学習モデル構築（scikit-learn）
### 講義時間：20分

---

> **講師メモ**：機械学習は「アルゴリズムの仕組み」より「ワークフローの流れ」を優先して教える。  
> 最重要ポイントは2つ：①「scaler.fit は訓練データだけ」②「混同行列の読み方」。  
> ランダムフォレストは「多数決」という一言で説明し、数式の詳細には踏み込まない。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target
df['品種'] = df['target'].map({0: 'Barolo', 1: 'Grignolino', 2: 'Barbera'})

print('準備完了')

---
## 【スライド1】機械学習の全体ワークフロー

### 今日体験する5ステップ

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.axis('off')

steps = [
    ('① データ準備\n・分割', 'train_test_split', '#4A90D9'),
    ('② 前処理\n（スケーリング）', 'StandardScaler', '#5BA85A'),
    ('③ モデル学習', 'model.fit()', '#E8A838'),
    ('④ 予測', 'model.predict()', '#C0392B'),
    ('⑤ 評価', 'accuracy_score\nconfusion_matrix', '#8E44AD'),
]

for i, (label, code, color) in enumerate(steps):
    x = 0.01 + i * 0.198
    box = mpatches.FancyBboxPatch((x, 0.1), 0.175, 0.82,
        boxstyle='round,pad=0.02', facecolor=color, edgecolor='white', lw=2,
        transform=ax.transAxes)
    ax.add_patch(box)
    ax.text(x + 0.0875, 0.80, label, transform=ax.transAxes,
            ha='center', va='top', fontsize=10, color='white', fontweight='bold')
    ax.text(x + 0.0875, 0.30, code, transform=ax.transAxes,
            ha='center', va='center', fontsize=8.5, color='#FDFEFE',
            fontfamily='monospace')
    if i < 4:
        ax.annotate('', xy=(x + 0.198, 0.51), xytext=(x + 0.178, 0.51),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='#444', lw=2))

ax.set_title('機械学習ワークフロー', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

---
## 【スライド2】なぜデータを「分割」するのか

### テストデータ = 「答え合わせ用」の未知のデータ

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')

# 全体データバー
bar = mpatches.FancyBboxPatch((0.03, 0.55), 0.94, 0.35,
    boxstyle='round,pad=0.01', facecolor='#D5D8DC', edgecolor='#999', lw=1,
    transform=ax.transAxes)
ax.add_patch(bar)
ax.text(0.50, 0.73, '全データ（178件）', transform=ax.transAxes,
        ha='center', va='center', fontsize=11, color='#555')

# 訓練データ
train = mpatches.FancyBboxPatch((0.03, 0.1), 0.74, 0.35,
    boxstyle='round,pad=0.01', facecolor='#4A90D9', edgecolor='white', lw=2,
    transform=ax.transAxes)
ax.add_patch(train)
ax.text(0.40, 0.275, '訓練データ（80% = 142件）\nmodel.fit() で使う', transform=ax.transAxes,
        ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# テストデータ
test = mpatches.FancyBboxPatch((0.78, 0.1), 0.19, 0.35,
    boxstyle='round,pad=0.01', facecolor='#E8A838', edgecolor='white', lw=2,
    transform=ax.transAxes)
ax.add_patch(test)
ax.text(0.875, 0.275, 'テスト\n（20%=36件）\n評価専用', transform=ax.transAxes,
        ha='center', va='center', fontsize=9, color='white', fontweight='bold')

ax.set_title('データ分割（Hold-out法）：80% 訓練 / 20% テスト', fontsize=13, pad=10)
plt.tight_layout()
plt.show()

print('なぜ分けるか？')
print('→ 「覚えたデータを答え合わせに使う」と、本当の実力がわからない')
print('→ 試験でいえば「練習問題 = 本番問題」にならないようにするため')

In [ ]:
# デモ：train_test_split
X = df[wine.feature_names]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('分割結果：')
print(f'  訓練データ: {X_train.shape[0]} 件')
print(f'  テストデータ: {X_test.shape[0]} 件')
print()
print('stratify=y の意味：品種の割合を保ったまま分割する')
print('  訓練データの品種分布:', dict(y_train.value_counts().sort_index()))
print('  テストデータの品種分布:', dict(y_test.value_counts().sort_index()))

---
## 【スライド3】なぜスケーリングが必要か

### 数値のスケール（大きさ）の違いが問題を起こす

In [ ]:
# デモ：スケールの差を可視化
cols_demo = ['alcohol', 'ash', 'proline', 'color_intensity']
stats = X_train[cols_demo].describe().loc[['min', 'max']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左：スケーリング前
X_train[cols_demo].boxplot(ax=axes[0])
axes[0].set_title('スケーリング前：値の範囲がバラバラ')
axes[0].set_ylabel('値')
axes[0].tick_params(axis='x', rotation=15)

# 右：スケーリング後
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_df = pd.DataFrame(X_train_scaled, columns=wine.feature_names)
X_train_df[cols_demo].boxplot(ax=axes[1])
axes[1].set_title('スケーリング後：すべて平均0・標準偏差1')
axes[1].set_ylabel('標準化された値')
axes[1].tick_params(axis='x', rotation=15)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5, label='平均=0')
axes[1].legend()

plt.suptitle('StandardScaler：標準化の効果', fontsize=13)
plt.tight_layout()
plt.show()

print('重要ルール：scaler.fit() は訓練データのみ！')
print('           テストデータには scaler.transform() だけを使う')
print()
print('理由：テストデータの情報をモデル構築に使ってはいけない（データリーク）')

---
## 【スライド4】ランダムフォレストとは

### 「たくさんの決定木の多数決」

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.axis('off')

# 決定木を3本表示（簡略図）
tree_colors = ['#4A90D9', '#5BA85A', '#E8A838']
tree_preds = ['Barolo', 'Barolo', 'Grignolino']

for i, (color, pred) in enumerate(zip(tree_colors, tree_preds)):
    x_center = 0.15 + i * 0.25

    # 木の図（簡略）
    trunk = mpatches.FancyBboxPatch((x_center - 0.08, 0.45), 0.16, 0.45,
        boxstyle='round,pad=0.015', facecolor=color, edgecolor='white', lw=2,
        alpha=0.85, transform=ax.transAxes)
    ax.add_patch(trunk)
    ax.text(x_center, 0.68, f'決定木 {i+1}', transform=ax.transAxes,
            ha='center', fontsize=11, color='white', fontweight='bold')

    # 予測結果
    result_color = '#C0392B' if pred == 'Grignolino' else '#2471A3'
    ax.text(x_center, 0.27, f'予測: {pred}', transform=ax.transAxes,
            ha='center', fontsize=10, color=result_color, fontweight='bold')

    # 矢印
    ax.annotate('', xy=(x_center, 0.30), xytext=(x_center, 0.43),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# 多数決
vote_box = mpatches.FancyBboxPatch((0.72, 0.3), 0.24, 0.38,
    boxstyle='round,pad=0.02', facecolor='#8E44AD', edgecolor='white', lw=2,
    transform=ax.transAxes)
ax.add_patch(vote_box)
ax.text(0.84, 0.56, '多数決', transform=ax.transAxes,
        ha='center', fontsize=12, color='white', fontweight='bold')
ax.text(0.84, 0.42, '最終予測\nBarolo', transform=ax.transAxes,
        ha='center', fontsize=11, color='#FDFEFE')

# データ → 各木への矢印
data_box = mpatches.FancyBboxPatch((0.00, 0.62), 0.12, 0.28,
    boxstyle='round,pad=0.015', facecolor='#D5D8DC', edgecolor='#999', lw=1,
    transform=ax.transAxes)
ax.add_patch(data_box)
ax.text(0.06, 0.76, '入力\nデータ', transform=ax.transAxes,
        ha='center', fontsize=9, color='#555')

for i in range(3):
    x_center = 0.15 + i * 0.25
    ax.annotate('', xy=(x_center - 0.06, 0.70), xytext=(0.12, 0.76),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='#888', lw=1))

# 各木 → 多数決への矢印
for i in range(3):
    x_center = 0.15 + i * 0.25
    ax.annotate('', xy=(0.72, 0.45), xytext=(x_center + 0.08, 0.27),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='#888', lw=1))

ax.set_title('ランダムフォレスト：多数の決定木の多数決で予測する', fontsize=13, pad=10)
plt.tight_layout()
plt.show()

print('ポイント：')
print('  ・各決定木はランダムに選んだデータとランダムな特徴量で学習する')
print('  ・バラバラな木を組み合わせることで、過学習を防ぐ')
print('  ・n_estimators（木の本数）を増やすと、より安定した結果が得られる')

In [ ]:
# デモ：モデル学習
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f'正解率（Accuracy）: {acc:.1%}')

---
## 【スライド5】評価指標：正解率だけでは不十分な理由

### 混同行列（Confusion Matrix）の読み方

In [ ]:
# 混同行列の読み方を説明
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：概念図（2クラスの場合）
ax = axes[0]
ax.axis('off')

concept = np.array([[8, 2], [1, 9]])
im = ax.imshow(concept, cmap='Blues', vmin=0, vmax=12)

labels = [['TP（正解！）\n正 → 正と予測', 'FP（誤検知）\n負 → 正と予測'],
          ['FN（見逃し）\n正 → 負と予測', 'TN（正解！）\n負 → 負と予測']]
colors = [['white', '#C0392B'], ['#E67E22', 'white']]

for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{concept[i,j]}\n\n{labels[i][j]}',
                ha='center', va='center', fontsize=9,
                color='black' if concept[i,j] > 6 else 'white')

ax.set_xticks([0, 1])
ax.set_xticklabels(['予測: 正', '予測: 負'], fontsize=11)
ax.set_yticks([0, 1])
ax.set_yticklabels(['実際: 正', '実際: 負'], fontsize=11)
ax.set_title('混同行列の読み方（2クラス）', fontsize=12)

ax.text(0, -0.7, '対角成分（TP + TN）= 正解した件数', transform=ax.transData,
        ha='left', fontsize=9, color='#2471A3')

# 右：実データ
cm = confusion_matrix(y_test, y_pred)
class_names = ['Barolo\n(0)', 'Grignolino\n(1)', 'Barbera\n(2)']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('ワインデータの混同行列')
axes[1].set_xlabel('予測ラベル')
axes[1].set_ylabel('正解ラベル')

plt.suptitle('混同行列（Confusion Matrix）', fontsize=13)
plt.tight_layout()
plt.show()

print('ポイント：正解率が高くても、誤分類のパターンを見ることが重要')
print('  FP（誤検知）を減らしたい場合 → Precision（適合率）を重視')
print('  FN（見逃し）を減らしたい場合 → Recall（再現率）を重視')

---
## 【スライド6】正解率だけでは不十分な例

### 不均衡データの落とし穴

In [ ]:
# 極端な例：1000件中990件が「陰性」のデータ
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.axis('off')

scenario = [
    ('データ', '陰性 990件\n陽性 10件', '#4A90D9'),
    ('「全部陰性」\nと予測するモデル', '正解率 = 99%\n（本当にOK？）', '#E8A838'),
    ('実際の問題', '陽性10件を\n全部見逃している！', '#C0392B'),
    ('正しい評価', 'Recall（再現率）= 0%\n→ このモデルは使えない', '#8E44AD'),
]

for i, (title, body, color) in enumerate(scenario):
    x = 0.02 + i * 0.245
    box = mpatches.FancyBboxPatch((x, 0.1), 0.21, 0.82,
        boxstyle='round,pad=0.02', facecolor=color, edgecolor='white', lw=2,
        transform=ax.transAxes)
    ax.add_patch(box)
    ax.text(x + 0.105, 0.82, title, transform=ax.transAxes,
            ha='center', va='top', fontsize=10, color='white', fontweight='bold')
    ax.text(x + 0.105, 0.38, body, transform=ax.transAxes,
            ha='center', va='center', fontsize=10, color='#FDFEFE')
    if i < 3:
        ax.annotate('', xy=(x + 0.244, 0.51), xytext=(x + 0.214, 0.51),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='#555', lw=2))

ax.set_title('正解率（Accuracy）だけでは不十分な理由', fontsize=13, pad=10)
plt.tight_layout()
plt.show()

print('医療検査・詐欺検出・不良品検査 など「見逃し」が致命的な場面では')
print('Recall（再現率）や F1 Score を必ず確認する必要があります。')

---
## 【スライド7】特徴量重要度の確認

### 「何が予測に効いているか」を可視化する

In [ ]:
# デモ：特徴量重要度
importances = pd.Series(model.feature_importances_, index=wine.feature_names)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#C0392B' if imp > 0.1 else 'steelblue' for imp in importances]
importances.plot(kind='barh', ax=ax, color=colors)
ax.axvline(x=0.1, color='red', linestyle='--', alpha=0.5, label='重要度 = 0.1')
ax.set_title('特徴量重要度（ランダムフォレスト）')
ax.set_xlabel('重要度')
ax.legend()
plt.tight_layout()
plt.show()

top_feature = importances.idxmax()
top_val = importances.max()
print(f'最も重要な特徴量: {top_feature}（重要度: {top_val:.3f}）')
print()
print('Ch.2のEDAで「プロリンとアルコールが品種を分けやすい」と仮説を立てました。')
print('→ 特徴量重要度がその仮説を検証してくれています！')

---
## 【スライド8】まとめ

### Ch.4 で覚えておくこと

| ステップ | コード | 注意点 |
|---------|--------|--------|
| データ分割 | `train_test_split(..., stratify=y)` | stratify でクラス比を保つ |
| スケーリング | `scaler.fit_transform(X_train)` | `fit` は訓練データのみ |
| テスト変換 | `scaler.transform(X_test)` | `fit_transform` は使わない |
| モデル学習 | `model.fit(X_train_scaled, y_train)` | — |
| 予測 | `model.predict(X_test_scaled)` | — |
| 評価 | `accuracy_score` + `confusion_matrix` | 必ずセットで確認する |

### 最重要ポイント2つ

> 1. **`scaler.fit` は訓練データのみ** ← データリーク防止
> 2. **正解率だけでなく混同行列も必ず確認** ← 誤分類のパターンが重要

### 次は「総合演習」へ
> 新しいデータ（乳がん診断データ）を使って、Ch.1〜4の流れを自力で実施します。